## PseudotimeDE Overview

In many single-cell analyses we might be interested in how gene expression changes across continuously varying cell states, like developmental trajectories or the cell cycle. It has become common practice to first infer _pseudotime_ coordinates associated with each cell, which can serve as a proxy for the development or cell cycle stage, for example. Second, gene expression can be tested for associations with these inferred pseudotimes. However, the pseudotime estimates are themselves subject to measurement uncertainty, and traditional hypothesis testing methods don't account for this. The PseudotimeDE algorithm provides a principled approach for accounting for this uncertainty when identifying genes with differential expression across a pseudotime trajectory. This notebook walks through an application of the method to mouse dendritic cells after lipopolysaccharide stimulation (GEO GSE45719). The data have a strong, genuine pseudotime effect, and we will find many genes significant even after accounting for estimation uncertainty.

This example dataset and the overall algorithm are borrowed from the PseudotimeDE R [package]((https://github.com/SONGDONGYUAN1994/PseudotimeDE/)) and [vignette]((https://github.com/SONGDONGYUAN1994/PseudotimeDE/blob/master/vignettes/quickstart.Rmd)), the main difference is that the underlying simulation engine has been replaced with scDesigner. This speeds up the estimation and allows us to work with all 100 genes present in the filtered data rather than just the two (CCL5 and CXCL10) that are studied in the original vignette. 

### Setup

We load all the packages that we'll use throughout this vignette and set some important hyperparameters. The number of subsamples controls the smallest possible p-value. The original PseudotimeDE paper recommends 1,000 but found relatively little empirical difference when using 100, which is what we will use in this notebook. Degree and DF (degrees of freedom) control the smoothness of the underlying spline models. Larger values allow for more flexibility but risk overfitting. To choose these hyperparameters it's best to visualize the fitted trajectories to see whether they roughly fit the data under the null and alternative models. We will demonstrate this below.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scdesigner import datasets

from pseudotimede_py import (
    fit_likelihood_improvement,
    run_null_fits,
    subsample_pseudotimes,
    summarize_calibration,
)

DEMO_GENES = ["CCL5", "CXCL10"]
N_SUBSAMPLES = 100
DF = 5
DEGREE = 3
SEED = 0

To use PseudotimeDE, it's necessary to first infer the pseudotimes on many subsamples of the data. This is what gives the algorithm a chance to evaluate the uncertainty induced by pseudotime estimation. Below, we have precomputed the trajectories from 100 runs of the slingshot R package, each using 80% of the available cells. The inferred pseudotimes are stored them in a pandas DataFrame given by the`.obsm["pseudotime_subsamples"]` slot of the overall anndata experiment object. Each row corresponds to one cell and each column to a different subsample. If a cell was not contained within one of the subsamples, we expect an NA. The `subsample_pseudotimes` function reorganizes this matrix into a list of Series (one per subsample).

In [ ]:
adata = datasets.lps()
subsamples = subsample_pseudotimes(adata)
genes = list(adata.var_names)
print(adata)
print(f"{len(subsamples)} subsamples, {len(subsamples[0])} cells each")

We first visualize the two demo genes before doing any testing. Qualitatively, there does seem to be some variation in the gene expression for these genes across the inferred pseudotime trajectories.

In [ ]:
pseudotime = adata.obs["pseudotime"]
counts = pd.DataFrame(
    adata.layers["counts"].toarray(), index=adata.obs_names, columns=adata.var_names
)

fig, axes = plt.subplots(
    1, 1 + len(DEMO_GENES), figsize=(4 * (1 + len(DEMO_GENES)), 3.5)
)
axes[0].hist(pseudotime, bins=30, color="black")
axes[0].set_xlabel("Pseudotime")
axes[0].set_ylabel("Number of cells")
for ax, gene in zip(axes[1:], DEMO_GENES):
    ax.scatter(pseudotime, np.log1p(counts[gene]), s=8, color="black", alpha=0.6)
    ax.set_xlabel("Pseudotime")
    ax.set_ylabel(f"log(1 + {gene} count)")
#fig.tight_layout()
#fig.show()

### PseudotimeDE Method

The scatterplot below visualizes how pseudo-time estimates change across subsamples. The x-axis gives the originally estimated pseudo-time and the y-axis gives estimates across subsamples (color). Points that are far from the diagonal are cells whose estimated pseudotimes are unstable.

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
for sub in subsamples[:5]:
    ax.scatter(pseudotime.loc[sub.index], sub, s=4, alpha=0.4)
ax.plot([0, 1], [0, 1], color="#C73333", linewidth=1)
ax.set_xlabel("Original pseudotime")
ax.set_ylabel("Subsample pseudotime")
fig.tight_layout()
fig.show()

As a measure of how important pseudotime is on gene expression, we compute the genewise log-likelihood ratios between models fit with and without pseudotime as a predictor. This is returned in the `statistic` column as $2 (\ell_{\text{spline}} - \ell_{\text{intercept}})$. The factor of two is conventional for likelihood comparisons; we do not assume chi-square calibration for this adaptation. Large values indicate stronger evidence of expression variation along the supplied pseudotime.

In [ ]:
observed, observed_models = fit_likelihood_improvement(
    adata, pseudotime, genes, df=DF, degree=DEGREE, return_models=True
)
observed.head()

#### Inspect the fits before testing

These are the exact models used for the observed statistics, not separately refitted showcase genes. We compare the intercept-only null and spline alternative with counts (black) and equal-width bin means (blue diamonds). Predictions on the grid reuse the training spline knots. The plotted response is the actual training `X`, which must contain untransformed counts for this analysis.

`scdiagnostics.plot_fitted_curve` accepts ordinary tidy tables, so it also works with other regression families and covariates. If needed, install the local package in this notebook's environment with `%pip install --no-deps -e ../../../scdiagnostics` (from the examples directory, with Altair and the other package dependencies already installed).

Shading is a **90% pointwise NB predictive interval for individual counts**, conditional on fitted parameters and supplied pseudotime, not a confidence band for the mean. Averaging is on the count scale before applying log1p. Thus the line is log(1 + fitted mean), not the expected log-count. Bins are visual summaries, not another smoother used for inference.

In [ ]:
import altair as alt
from scdiagnostics import plot_fitted_curve
from pseudotimede_py.diagnostics import model_fit_tables

fit_counts, fit_curves, fit_residuals = model_fit_tables(observed_models, DEMO_GENES)
plot_fitted_curve(
    fit_counts, fit_curves, x="pseudotime", y="count",
    group="gene", model="model", lower="lower", upper="upper",
    transform=np.log1p, y_title="log(1 + count or mean)",
)

A flat null missing a real trend is expected. A spline missing the bin means, boundary spikes in sparsely sampled regions, or systematic residual trends are reasons to revisit the fit. Alternative Pearson residuals below use NB variance $\mu + \mu^2/r$. Look for trends and extreme observations, not normally distributed residuals: these residuals can be discrete and asymmetric. Constant dispersion can miss changes in zero frequency or tails even when the mean looks adequate.

These are descriptive in-sample checks, not formal goodness-of-fit tests or proof of calibrated p-values. Change the genes passed to `model_fit_tables` to inspect weak effects and flagged fits as well as the showcase genes. If spline or training settings change, rerun observed and permutation fits consistently; choosing flexibility after inspecting effects can itself affect calibration.

In [ ]:
residual_data = fit_residuals.query("model == 'alternative'")
base = alt.Chart(residual_data).encode(
    x=alt.X("pseudotime:Q"), y=alt.Y("pearson:Q", title="Pearson residual")
)
points = base.mark_circle(size=15, opacity=0.35).encode(tooltip=["cell:N", "pearson:Q"])
zero = base.mark_rule(color="#C73333").encode(x=alt.Undefined, y=alt.datum(0))
(points + zero).properties(width=300, height=200).facet(column="gene:N")

**Optimization is a separate check.** Loss is pooled across all genes: a plateau or `converged_*` flag does not establish per-gene convergence. Substantially negative likelihood improvements, nonfinite fits, or meaningful changes after longer training need investigation before interpreting p-values. A plausible curve alone cannot detect every optimizer failure. Inspect representative shuffled-subsample fits too; failed permutations must remain failures rather than being dropped to improve the null histogram.

In [ ]:
loss_history = pd.concat([
    pd.DataFrame(model.fit_history).assign(model=name)
    for name, model in observed_models.items()
], ignore_index=True)
alt.Chart(loss_history).mark_line().encode(
    x="epoch:Q", y=alt.Y("train_loss:Q", scale=alt.Scale(zero=False)),
    color="model:N",
).properties(width=600, height=180)

Instead of trusting the asymptotic approximation, PseudotimeDE evaluates the significance of the test statistic $T$ using a permutation null. For each subsample we randomly permute the inferred pseudotimes. This breaks any relationship between the observed gene expression values and pseudotime. Recomputing the test statistic $T$ for each gene on these permuted samples gives its reference null distribution. This step can be time-consuming because it needs to fit both null and spline models for every subsample. The example below took X minutes for 100 genes and 100 subsamples.


In [ ]:
start = time.perf_counter()
null = run_null_fits(
    adata, subsamples[:N_SUBSAMPLES], genes, seed=SEED, df=DF, degree=DEGREE
)
print(f"{N_SUBSAMPLES} subsamples, {len(genes)} genes: {time.perf_counter() - start:.0f} s")
null.head()

Values of $T$ that lie in the tail of the null distribution above are evidence of a true pseudotime effect, and they can be used to compute an empirical p-value $(1 + \#\{T_{\text{null}} \geq T_{\text{obs}}\}) / (B + 1)$. This formula shows why larger subsamples $B$ can provide more accurate p-values for strong effects. 

In [ ]:
results = summarize_calibration(observed["statistic"], null)
minimum = 1 / (N_SUBSAMPLES + 1)
results.loc[DEMO_GENES]

### Diagnostics

We visualize this permutation $p$-value for the demo genes in the plot below. The permutation null is the black background histogram and the observed test statistics are vertical red lines.

In [ ]:
fig, axes = plt.subplots(
    1, len(DEMO_GENES), figsize=(4.5 * len(DEMO_GENES), 3.5), squeeze=False
)
for ax, gene in zip(axes[0], DEMO_GENES):
    gene_null = null.xs(gene, level="gene")
    ax.hist(gene_null.loc[gene_null["fit_ok"], "statistic"], bins=20, color="black")
    ax.axvline(observed.loc[gene, "statistic"], color="#C73333", linewidth=1)
    ax.set_xlabel("Likelihood-improvement statistic")
    ax.set_ylabel("Number of permutations")
    ax.set_title(f"{gene} (p = {results.loc[gene, 'p_empirical']:.3f})")
fig.tight_layout()
fig.show()